In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
import re
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor

INPUT_FILE = Path("Копия done_diplom_filled_v2_with_keyrate_income_filled.xlsx")
SHEET_NAME = "merged_mortgage_prices_quarter"
OUTPUT_XLSX = Path("excel_file1.xlsx")
OUTPUT_TXT = Path("txt_file1.txt")

ALPHA = 0.05
MAX_LAG = 12
HAC_LAGS = 12

# как и в части прогнозов кажется проще перевести все имена в более короткие чем хранить в огромных названиях
def safe_colname(col: str) -> str:
    mapping = {
        "Дата": "date",
        "Задолженность по предоставленным кредитам, млн руб., в том числе": "total_debt",
        "просроченная задолженность по предоставленным кредитам, млн руб.": "overdue_debt",
        "Средневзвешенный срок кредитования по кредитам, выданным в течение месяца, месяцев": "avg_term_months",
        "Средневзвешенная ставка по кредитам, выданным в течение месяца, %": "avg_mortgage_rate",
        "EarlyRep: Объем досрочно погашенных ипотечных жилищных кредитов (прав требования по ипотечным жилищным кредитам) | выданных (приобретенных) в рублях | средствами заемщика": "earlyrep_borrower",
        "EarlyRep: вновь выданными ипотечными жилищными кредитами": "earlyrep_new_loans",
        "EarlyRep: средствами, полученными от реализации заложенного имущества": "earlyrep_collateral",
        "Refin: Объем рефинансированных ИЖК с продажей пула ИЖК (прав требования) | всего | в рублях": "refinancing",
        "Цена_квм: Низкого качества | Вторичный рынок жилья": "price_secondary_low",
        "Цена_квм: Квартиры среднего качества (типовые) | Вторичный рынок жилья": "price_secondary_typical",
        "Цена_квм: Квартиры среднего качества (типовые) | Первичный рынок жилья": "price_primary_typical",
        "Цена_квм: Улучшенного качества | Вторичный рынок жилья": "price_secondary_improved",
        "Цена_квм: Улучшенного качества | Первичный рынок жилья": "price_primary_improved",
        "Инфляция_%_мм": "infl_mom",
        "Инфляция_%_гг": "infl_yoy",
        "ИПЦ_%_к_пред_мес_индекс": "cpi_index_mom",
        "Занятые_15_72_тыс": "employed_ths",
        "Уровень_занятости_15_72_%": "employment_rate",
        "Безработные_15_72_тыс": "unemployed_ths",
        "Уровень_безработицы_15_72_%": "unemployment_rate",
        "ЗП_номин_руб": "nominal_wage",
        "Ключевая ставка, % годовых": "key_rate",
    }
    col = str(col).strip()
    if col in mapping:
        return mapping[col]
    col = re.sub(r"[^0-9a-zA-Zа-яА-Я_]+", "_", col).strip("_").lower()
    return col or "unnamed"


def parse_avg_mortgage_rate(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (pd.Timestamp, datetime)):
        return float(f"{x.month}.{x.day:02d}")
    try:
        return float(x)
    except Exception:
        return np.nan


def build_housing_index(df: pd.DataFrame, cols: list[str], base: float = 100.0) -> pd.Series:
    temp = df[cols].astype(float).ffill()
    return temp.divide(temp.iloc[0]).mean(axis=1) * base


def adf_pvalue(s: pd.Series) -> float:
    s = pd.Series(s).dropna()
    if len(s) < 12:
        return np.nan
    try:
        return adfuller(s, autolag="AIC")[1]
    except Exception:
        return np.nan


def kpss_pvalue(s: pd.Series) -> float:
    s = pd.Series(s).dropna()
    if len(s) < 12:
        return np.nan
    try:
        return kpss(s, regression="c", nlags="auto")[1]
    except Exception:
        return np.nan

# как я понимаю уровни значимости именно так обозначаются в науч литературе
def significance_stars(p):
    if pd.isna(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""


def direction_label(coef):
    if pd.isna(coef):
        return ""
    return "positive" if coef > 0 else "negative"


def fit_hac_ols(y: pd.Series, X: pd.DataFrame, hac_lags: int = HAC_LAGS):
    return sm.OLS(y, sm.add_constant(X)).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})


def vif_table(X: pd.DataFrame) -> pd.DataFrame:
    Xv = sm.add_constant(X)
    rows = []
    for i, col in enumerate(Xv.columns):
        if col == "const":
            continue
        rows.append([col, variance_inflation_factor(Xv.values, i)])
    return pd.DataFrame(rows, columns=["variable", "vif"])


def make_coef_table(model):
    return pd.DataFrame({"variable": model.params.index,"coef": model.params.values,"std_err": model.bse.values,"t_value": model.tvalues.values,"p_value": model.pvalues.values,
        "ci_low": model.conf_int().iloc[:, 0].values,"ci_high": model.conf_int().iloc[:, 1].values,})


def make_metrics(model, X, resid=None):
    resid = model.resid if resid is None else resid
    lb6 = acorr_ljungbox(resid, lags=[6], return_df=True)["lb_pvalue"].iloc[0]
    lb12 = acorr_ljungbox(resid, lags=[12], return_df=True)["lb_pvalue"].iloc[0]
    bp_p = het_breuschpagan(resid, sm.add_constant(X))[1]
    jb_p = jarque_bera(resid)[1]
    dw = sm.stats.stattools.durbin_watson(resid)
    return pd.DataFrame({
        "metric": ["R^2", "Adj. R^2", "AIC", "BIC", "DW", "Ljung-Box p (lag 6)", "Ljung-Box p (lag 12)", "Breusch-Pagan p", "Jarque-Bera p"],
        "value": [model.rsquared, model.rsquared_adj, model.aic, model.bic, dw, lb6, lb12, bp_p, jb_p],
    })


def build_dataset(y, regspec: dict):
    data = {"y": y}
    for name, (series, lag) in regspec.items():
        data[name] = series.shift(lag)
    return pd.DataFrame(data).dropna()


def evaluate_model(label, y, regspec: dict):
    ds = build_dataset(y, regspec)
    model = fit_hac_ols(ds["y"], ds.drop(columns="y"), hac_lags=HAC_LAGS)
    coef = make_coef_table(model)
    metrics = make_metrics(model, ds.drop(columns="y"))
    vif = vif_table(ds.drop(columns="y"))
    return {"label": label, "dataset": ds, "model": model, "coef": coef, "metrics": metrics, "vif": vif}


raw = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)
raw.columns = [safe_colname(c) for c in raw.columns]
if "avg_mortgage_rate" in raw.columns:
    raw["avg_mortgage_rate"] = raw["avg_mortgage_rate"].apply(parse_avg_mortgage_rate)
for c in raw.columns:
    if c == "date":
        raw[c] = pd.to_datetime(raw[c], errors="coerce")
    else:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")
raw = raw.sort_values("date").reset_index(drop=True).ffill()

df = pd.DataFrame({"date": raw["date"]})
df["total_debt"] = raw["total_debt"]
df["overdue_debt"] = raw["overdue_debt"]
df["target_ratio"] = raw["overdue_debt"] / raw["total_debt"]
df["avg_term_months"] = raw["avg_term_months"]
df["avg_mortgage_rate"] = raw["avg_mortgage_rate"]
df["key_rate"] = raw["key_rate"]
df["spread_rate"] = df["key_rate"] - df["avg_mortgage_rate"]
df["housing_index_primary"] = build_housing_index(raw, ["price_primary_typical", "price_primary_improved"])
df["housing_index_secondary"] = build_housing_index(raw, ["price_secondary_low", "price_secondary_typical", "price_secondary_improved"])
df["infl_mom"] = raw["infl_mom"]
df["infl_yoy"] = raw["infl_yoy"]
df["cpi_index_mom"] = raw["cpi_index_mom"]
df["unemployed_ths"] = raw["unemployed_ths"]
df["unemployment_rate"] = raw["unemployment_rate"]
df["nominal_wage"] = raw["nominal_wage"]
df["earlyrep_borrower"] = raw["earlyrep_borrower"]
df["earlyrep_new_loans"] = raw["earlyrep_new_loans"]
df["earlyrep_collateral"] = raw["earlyrep_collateral"]
df["refinancing"] = raw["refinancing"]
clean_data = df.copy()

transformed = pd.DataFrame({"date": df["date"]})
transformed["target_ratio"] = np.log(df["target_ratio"]).diff(12).diff()
transformed["avg_term_months"] = np.log(df["avg_term_months"]).diff()
transformed["avg_mortgage_rate"] = df["avg_mortgage_rate"].diff()
transformed["earlyrep_borrower"] = np.log(df["earlyrep_borrower"]).diff()
transformed["earlyrep_new_loans"] = np.log(df["earlyrep_new_loans"]).diff()
transformed["earlyrep_collateral"] = np.log(df["earlyrep_collateral"]).diff()
transformed["refinancing"] = np.log(df["refinancing"]).diff()
transformed["housing_index_primary"] = np.log(df["housing_index_primary"]).diff()
transformed["housing_index_secondary"] = np.log(df["housing_index_secondary"]).diff()
transformed["infl_mom"] = df["infl_mom"]
transformed["infl_yoy"] = df["infl_yoy"].diff()
transformed["cpi_index_mom"] = df["cpi_index_mom"]
transformed["unemployed_ths"] = np.log(df["unemployed_ths"]).diff()
transformed["unemployment_rate"] = df["unemployment_rate"].diff()
transformed["nominal_wage"] = np.log(df["nominal_wage"]).diff()
transformed["key_rate"] = df["key_rate"].diff()
transformed["spread_rate"] = df["spread_rate"].diff()

transform_map = {
    "target_ratio": ("ΔΔ12 ln(target)", "сделали раньше так"),
    "avg_term_months": ("Δln(x)", "log-difference"),
    "avg_mortgage_rate": ("Δx", "first difference"),
    "earlyrep_borrower": ("Δln(x)", "log-difference"),
    "earlyrep_new_loans": ("Δln(x)", "log-difference"),
    "earlyrep_collateral": ("Δln(x)", "log-difference"),
    "refinancing": ("Δln(x)", "log-difference"),
    "housing_index_primary": ("Δln(x)", "log-difference"),
    "housing_index_secondary": ("Δln(x)", "log-difference"),
    "infl_mom": ("level", "stationary in levels"),
    "infl_yoy": ("Δx", "first difference"),
    "cpi_index_mom": ("level", "stationary in levels"),
    "unemployed_ths": ("Δln(x)", "log-difference"),
    "unemployment_rate": ("Δx", "first difference"),
    "nominal_wage": ("Δln(x)", "log-difference"),
    "key_rate": ("Δx", "first difference"),
    "spread_rate": ("Δx", "first difference"),}

stationarity = []
for var in transform_map:
    stationarity.append({
        "var": var,"adf_p_level": adf_pvalue(df[var]),"kpss_p_level": kpss_pvalue(df[var]),"chosen_transform": transform_map[var][0],
        "adf_p_trans": adf_pvalue(transformed[var]),"kpss_p_trans": kpss_pvalue(transformed[var]),"decision": transform_map[var][1],})
stationarity = pd.DataFrame(stationarity)

candidate_factors = [
    "avg_term_months", "spread_rate", "housing_index_primary", "housing_index_secondary",
    "infl_mom", "infl_yoy", "cpi_index_mom", "unemployed_ths", "unemployment_rate",
    "nominal_wage", "earlyrep_borrower", "earlyrep_new_loans", "earlyrep_collateral", "refinancing",
]

y = transformed["target_ratio"]

def scan_single_factor_all(y: pd.Series, x: pd.Series, factor: str, max_lag: int = MAX_LAG):
    rows = []
    for lag in range(max_lag + 1):
        temp = pd.DataFrame({"y": y, "y_l1": y.shift(1), "x_lag": x.shift(lag)}).dropna()
        if len(temp) < 20:
            continue
        model = fit_hac_ols(temp["y"], temp[["y_l1", "x_lag"]], hac_lags=HAC_LAGS)
        rows.append({
            "factor": factor,"lag": lag,"coef": model.params["x_lag"],"p": model.pvalues["x_lag"],"t": model.tvalues["x_lag"],
            "n": int(model.nobs),"adj_r2": model.rsquared_adj,"direction": direction_label(model.params["x_lag"]),"signif": significance_stars(model.pvalues["x_lag"]),"y_l1_p": model.pvalues["y_l1"],})
    return pd.DataFrame(rows)

all_univariate_rows = []
best_rows = []
for factor in candidate_factors:
    all_lags = scan_single_factor_all(y, transformed[factor], factor, MAX_LAG)
    if all_lags.empty:
        continue
    all_univariate_rows.append(all_lags)
    best = all_lags.sort_values(["p", "adj_r2"], ascending=[True, False]).iloc[0]
    best_rows.append(best)

univariate_all_lags = pd.concat(all_univariate_rows, ignore_index=True)
univariate_best_lags = pd.DataFrame(best_rows).sort_values(["p", "adj_r2"], ascending=[True, False]).reset_index(drop=True)


best_lag_map = {r["factor"]: int(r["lag"]) for _, r in univariate_best_lags.iterrows()}
expanded_regspec = {
    "y_l1": (transformed["target_ratio"], 1),
    "y_l2": (transformed["target_ratio"], 2),
}
for factor in candidate_factors:
    expanded_regspec[f"{factor}_l{best_lag_map[factor]}"] = (transformed[factor], best_lag_map[factor])

expanded = evaluate_model("лушчие_lags", y, expanded_regspec)

reduction_steps = []

def record_step(step_no, removed, reason, result):
    pmap = result["model"].pvalues.to_dict()
    reduction_steps.append({
        "step": step_no,"removed": removed,"reason": reason,"nobs": int(result["model"].nobs),"adj_r2": result["model"].rsquared_adj,
        "aic": result["model"].aic,"bic": result["model"].bic,"spread_p": pmap.get("spread_rate_l12", np.nan),"key_rate_p": pmap.get("key_rate_l12", np.nan),"infl_yoy_p": pmap.get("infl_yoy_l6", np.nan),"unemployment_rate_p": pmap.get("unemployment_rate_l12", np.nan),
        "earlyrep_collateral_p": pmap.get("earlyrep_collateral_l9", np.nan),"max_p_nonconst": max([v for k, v in pmap.items() if k != "const"], default=np.nan),"variables": ", ".join([c for c in result["coef"]["variable"].tolist() if c != "const"]),})

record_step(0, "", "расширенная модель", expanded)

step1_regspec = {
    "y_l1": (transformed["target_ratio"], 1),
    "y_l2": (transformed["target_ratio"], 2),"spread_rate_l12": (transformed["spread_rate"], 12),"key_rate_l12": (transformed["key_rate"], 12),
    "unemployment_rate_l12": (transformed["unemployment_rate"], 12),"infl_yoy_l6": (transformed["infl_yoy"], 6),"earlyrep_collateral_l9": (transformed["earlyrep_collateral"], 9),
    "avg_term_months_l12": (transformed["avg_term_months"], 12),"housing_index_secondary_l4": (transformed["housing_index_secondary"], 4),"housing_index_primary_l4": (transformed["housing_index_primary"], 4),
    "earlyrep_borrower_l1": (transformed["earlyrep_borrower"], 1),"earlyrep_new_loans_l9": (transformed["earlyrep_new_loans"], 9),
    "infl_mom_l2": (transformed["infl_mom"], 2),"cpi_index_mom_l2": (transformed["cpi_index_mom"], 2),"unemployed_ths_l12": (transformed["unemployed_ths"], 12),}
step1 = evaluate_model("step1_dense_core", y, step1_regspec)
record_step(1, "", "модель только с значимыми факторами", step1)

step2_regspec = {
    "y_l1": (transformed["target_ratio"], 1),
    "y_l2": (transformed["target_ratio"], 2),
    "spread_rate_l12": (transformed["spread_rate"], 12),"key_rate_l12": (transformed["key_rate"], 12),"unemployment_rate_l12": (transformed["unemployment_rate"], 12),
    "infl_yoy_l6": (transformed["infl_yoy"], 6),
    "earlyrep_collateral_l9": (transformed["earlyrep_collateral"], 9),"avg_term_months_l12": (transformed["avg_term_months"], 12),"housing_index_secondary_l4": (transformed["housing_index_secondary"], 4),
    "earlyrep_borrower_l1": (transformed["earlyrep_borrower"], 1),"earlyrep_new_loans_l9": (transformed["earlyrep_new_loans"], 9),}
step2 = evaluate_model("убрать дубликаты", y, step2_regspec)
record_step(2, "cpi_index_mom_l2, infl_mom_l2, unemployed_ths_l12, housing_index_primary_l4", "тут поборолись с мультиколленеарностью", step2)


step3_regspec = {
    "y_l1": (transformed["target_ratio"], 1),
    "y_l2": (transformed["target_ratio"], 2),
    "spread_rate_l12": (transformed["spread_rate"], 12),"key_rate_l12": (transformed["key_rate"], 12),
    "unemployment_rate_l12": (transformed["unemployment_rate"], 12),"infl_yoy_l6": (transformed["infl_yoy"], 6),"earlyrep_collateral_l9": (transformed["earlyrep_collateral"], 9),}
step3 = evaluate_model("step3_core_with_key", y, step3_regspec)
record_step(3, "avg_term_months_l12, housing_index_secondary_l4, earlyrep_borrower_l1, earlyrep_new_loans_l9", "выкинули нестабильные фичи", step3)


final_regspec = {
    "y_l1": (transformed["target_ratio"], 1),
    "y_l2": (transformed["target_ratio"], 2),
    "spread_rate_l12": (transformed["spread_rate"], 12),
    "unemployment_rate_l12": (transformed["unemployment_rate"], 12),
    "infl_yoy_l6": (transformed["infl_yoy"], 6),
    "earlyrep_collateral_l9": (transformed["earlyrep_collateral"], 9),
}
final_result = evaluate_model("final_model", y, final_regspec)
record_step(4, "key_rate_l12", "они с спредом забирают друг у друга значимость поэтому этот фактор уходит ", final_result)

spread_vs_keyrate = pd.DataFrame([
    {"model": "without_key_rate","spread_p": final_result["model"].pvalues["spread_rate_l12"],"key_rate_p": np.nan,"adj_r2": final_result["model"].rsquared_adj,},
    {"model": "with_key_rate","spread_p": step3["model"].pvalues["spread_rate_l12"],"key_rate_p": step3["model"].pvalues["key_rate_l12"],"adj_r2": step3["model"].rsquared_adj,
    },])

model_path = pd.DataFrame(reduction_steps)
intermediate_models = []
for res in [expanded, step1, step2, step3, final_result]:
    c = res["coef"].copy()
    c.insert(0, "model_label", res["label"])
    intermediate_models.append(c)
intermediate_models = pd.concat(intermediate_models, ignore_index=True)

intermediate_metrics = []
for res in [expanded, step1, step2, step3, final_result]:
    m = res["metrics"].copy()
    m.insert(0, "model_label", res["label"])
    intermediate_metrics.append(m)
intermediate_metrics = pd.concat(intermediate_metrics, ignore_index=True)
"""
Код ниже был сгенерирован нейросетью deepseek
Также названия были скорректированы мною , то есть код сгенерированный с моими правками
Код был сгенерирован по промту : Вот часть моей  работы по ВКР в которой я строю модель для факторного анализа приведенных фичей .Сохрани мне это все красиво в несеолько файлов или в один файл эксель на разные страницы"

Модель: deepseek,expert,deepthing https://chat.deepseek.com

Работа выполнена правильно на 8 из 10
"""
lines = [
    "финальная модепльL\n",
    "само уравнение: ΔΔ12 ln(target_ratio) ~ AR(2) + Δspread_l12 + Δunemployment_l12 + Δinfl_yoy_l6 + Δln(earlyrep_collateral)_l9\n\n",
    "кэфы :\n"
]
for _, r in final_result["coef"].iterrows():
    lines.append(f"{r['variable']:30s} {r['coef']: 8.6f} (p={r['p_value']:.4f})\n")

lines.append(f"\nAdj. R² = {final_result['metrics'].loc[final_result['metrics']['metric']=='adj_r2','value'].values[0]:.4f}\n")
lines.append(f"спред без ключевой ставки  = {final_result['model'].pvalues['spread_rate_l12']:.6f}\n")
lines.append(f"спред с ключевофй ставкой     = {step3['model'].pvalues['spread_rate_l12']:.6f}\n")

OUTPUT_TXT.write_text("".join(lines), encoding="utf-8")
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    clean_data.to_excel(writer, sheet_name="clean_data", index=False)
    transformed.to_excel(writer, sheet_name="transformed_data", index=False)
    stationarity.to_excel(writer, sheet_name="stationarity", index=False)
    univariate_all_lags.to_excel(writer, sheet_name="univariate_all_lags", index=False)
    univariate_best_lags.to_excel(writer, sheet_name="univariate_best_lags", index=False)
    expanded["coef"].to_excel(writer, sheet_name="expanded_model_coef", index=False)
    expanded["metrics"].to_excel(writer, sheet_name="expanded_model_metrics", index=False)
    intermediate_models.to_excel(writer, sheet_name="intermediate_models_coef", index=False)
    intermediate_metrics.to_excel(writer, sheet_name="intermediate_models_metrics", index=False)
    model_path.to_excel(writer, sheet_name="model_reduction_path", index=False)
    final_result["coef"].to_excel(writer, sheet_name="final_model_coef", index=False)
    final_result["metrics"].to_excel(writer, sheet_name="final_model_metrics", index=False)
    final_result["vif"].to_excel(writer, sheet_name="final_model_vif", index=False)
    spread_vs_keyrate.to_excel(writer, sheet_name="spread_vs_keyrate", index=False)




Saved: time_series_factor_analysis_replication_enhanced_output.xlsx
Saved: time_series_factor_analysis_replication_enhanced_summary.txt
Final spread p without key_rate: 0.030339
Final spread p with key_rate: 0.264957


d